# Dataset creation

_________

In [1]:
import pandas as pd
import random
import re
import requests
from pathlib import Path
from datasets import load_dataset
import html
import nltk
from nltk.tokenize import sent_tokenize

random.seed(1985)

___

### Step 1: BBC news

In [318]:
# xsum chosen to replace bbc news as mentioned in report, due to both number of articles and average length of those articles

In [319]:
xsum = load_dataset("EdinburghNLP/xsum")
xsum

DatasetDict({
    train: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 204045
    })
    validation: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11332
    })
    test: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11334
    })
})

In [320]:
# xsum is in train/test/validation format but we are only interested in the data so we can concatenate these back together

xsum_df = pd.concat(
    [
        xsum["train"].to_pandas(),
        xsum["validation"].to_pandas(),
        xsum["test"].to_pandas()
    ],
    ignore_index=True
)

In [321]:
# xsum_df.head()

In [322]:
# rename and add columns to bring in line with other datasets
xsum_df = xsum_df.rename(columns={"document" : "text"})
xsum_df["source"] = "BBC"
xsum_df["topic"] = "General"
# NB: initially wanted N/A here but it was interpreted as NaN and I didnt want problems down the line

In [323]:
# get words counts and describe
xsum_df["word_count"] = (xsum_df["text"].str.split().str.len())
xsum_df["word_count"].describe()

count    226711.000000
mean        373.742143
std         303.724687
min           0.000000
25%         177.000000
50%         295.000000
75%         491.000000
max       29189.000000
Name: word_count, dtype: float64

In [324]:
# above confirms dataset articles are long enough, and enough variety for sampling randomly
# now trim down to those of suitable length

xsum_usable = xsum_df[
    (xsum_df["word_count"] >= 350)
    &
    (xsum_df["word_count"] <= 450)
].copy()

print("Usable articles: ", len(xsum_usable))

Usable articles:  27798


In [325]:
# randomly sample some articles to look out for any common issues, though with ~28000 articles this wil be brief

# pd.set_option("display.max_colwidth", None)
# xsum_usable.sample(5)

In [326]:
# I note the dataset uses \n tags so must remove these, and so will make a function to remove tags, extra whitespaces, urls etc. from text
# this covers all the issues found in each seperate dataset

def clean_text(text):

    # decode html, run twice for double encoding (lots in reddit dataset)
    text = html.unescape(html.unescape(text))

    # each gutenberg book has text about project Gutenberg before the book begins, which is typically marked by a tag such as:
    # *** START OF THIS PROJECT GUTENBERG EBOOK THE FIVE GIANTS *** there is also a similar *** END... tag. 
    start = re.search(r"\*\*\* START OF .*?\*\*\*", text, re.DOTALL)
    if start:
        text = text[start.end():]
        
    end = re.search(r"\*\*\* END OF .*?\*\*\*", text, re.DOTALL)
    if end:
        text = text[:end.start()]

    # remove href tags (seen in amazon dataset) and the <br /> tags in wiki dataset
    text = re.sub(r"<[^>]+>", " ", text)

    # reddit dataset often has markdown links, want to remove these but keep the text
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)
    
    # lots of broken markdown links in reddit ie [text](broken url
    text = re.sub(r"\[([^\]]+)\]\(", r"\1", text)
    
    # fix new line tags in xsum
    text = text.replace("\n", " ")

    # remove wikipedia section headings (== History == etc.)
    text = re.sub(r"=+\s*[^=]+\s*=+", " ", text)
        
    # remove urls
    text = re.sub(r"http\S+", "", text)

    # remove twitter usernames @johnsmith etc.
    text = re.sub(r"@\w+", "", text)

    # remove reddit usernames u/johnsmith etc.
    text = re.sub(r"u/\w+", "", text)

    # odd pieces of text across all formats can sometimes be broken and combine multiple words into one long word
    # in these cases i do not want to manually ammend, and I do not want to remove the long word as this impacts
    # the meaning of the text itself so in these cases i will not return the text
    words = text.split()
    if any(len(word) > 100 for word in words):
        return None
    
    # remove extra whtiespace
    text = re.sub(r"\s+", " ", text)

    # finally return text file minus whitespace on the ends
    return text.strip()   

In [327]:
# run the cleaning on xsum news dataset
xsum_usable["text"] = xsum_usable["text"].apply(clean_text)
# and then remove any rows where documents were rejected
xsum_usable = xsum_usable.dropna(subset = ["text"]).copy()

In [328]:
# check it worked as expected
# xsum_usable.sample(5)

In [329]:
xsum_usable = xsum_usable.sample(n = 400, random_state = 1985)

In [330]:
# add document ids (after trimming to 400)
xsum_usable["document_id"] = [

    # +1 as dont want '0', 4 digits as dataset potentially going up to 8000 entries
    f"BBC_{i+1:04d}"

    for i in range(len(xsum_usable))
]

In [331]:
xsum_usable = xsum_usable[
    [
        "document_id",
        "source",
        "topic",
        "text",
        "word_count"
    ]
]

In [332]:
xsum_usable["word_count"].describe()

count    400.000000
mean     395.207500
std       28.333751
min      350.000000
25%      371.000000
50%      394.000000
75%      418.000000
max      450.000000
Name: word_count, dtype: float64

In [335]:
# xsum_usable.sample(10)

In [336]:
xsum_usable.to_csv(r"C:\Users\Jamie\Desktop\DSM500 Final Project\Data\Processed\bbc_xsum_dataset.csv", index = False)

_________

### Step 2: Wikipedia

In [4]:
API_URL = "https://en.wikipedia.org/w/api.php"

HEADERS = {"User-Agent": "DSMScProject (james.a.east@gmail.com)"}

In [5]:
# define topics within wikipedia good articles
# NB' social sciences covers politics etc

topic_sources = {
    "Engineering and technology": "Wikipedia:Good articles/Engineering and technology",
    "History": "Wikipedia:Good articles/History",
    "Natural sciences": "Wikipedia:Good articles/Natural sciences",
    "Sports and recreation": "Wikipedia:Good articles/Sports and recreation",
    "Social sciences and society": "Wikipedia:Good articles/Social sciences and society"
}

In [6]:
# function to get the wikitext from a wikipedia page
# wiki text used before conversion to html
# this allows us to get article titles

def get_source_page_wiki_text(source_page):

    # set parameters
    params = {
        "action": "parse",
        "page": source_page,
        "prop": "wikitext",
        "format": "json"
    }

    # contact server
    response = requests.get(
        API_URL,
        params = params,
        headers = HEADERS
    )

    # convert that json response into python
    data = response.json()

    # return article text
    try:
        return data["parse"]["wikitext"]["*"] 

    # skip if error
    except KeyError:
        return ""

In [7]:
# function to get article titles from the above source page

def get_wiki_titles(wikitext):

    # wiki text has titles in [[title]] so looking for these
    titles = re.findall(r"\[\[(.*?)\]\]", wikitext)
        
    cleaned = []

    for title in titles:
    
        # get rid of display text
        title = title.split("|")[0]
        # remove links
        title = title.split("#")[0]
        title = title.strip()
    
        if title:
    
            cleaned.append(title)
    
    return list(set(cleaned))

In [8]:
# function to get article text

def get_article_text(title):

    # set parameters
    params = {
        "action":      "query",
        "prop":        "extracts",
        "explaintext": True,
        "titles":      title,
        "format":      "json"
    }

    # send request
    response = requests.get(
        API_URL,
        params = params,
        headers = HEADERS
    )

    # convert json
    data = response.json()
    # get pages from response
    pages = data["query"]["pages"]
    # get first page
    page = list(pages.values())[0]
    # if page has article, extract it
    if "extract" in page:
        return page["extract"]
    # otherwise return blank
    else:

        return ""

In [2]:
# wiki articles can be thousands of words and will likely not find any 'Good articles' that meet the 350-450
# requirement so we will cut off after 400 words

def get_sample(text):

    # split text in seperate words
    words = text.split()
    # ignore article if num of words < 350
    if len(words) < 350:
        return None

    # split article into sentences
    # this is not for nlp or pre-processing, just to get to end of sentence when hit 400 words
    sentences = sent_tokenize(text)

    # initialise 0 words and open array to store sentences
    sentences_chosen = []
    word_count = 0

    # loop through sentences and add to array whilst counting words
    for sentence in sentences:
        sentences_chosen.append(sentence)
        word_count += len(sentence.split())

        # when words >= 400 break out
        if word_count >= 400:
                break
        
    # reconnect sentences to make sample
    sample = " ".join(sentences_chosen)

    return sample

On inspection, some wikipedia articles are very techical and do not contain a great deal of syntactic content. To give an example of one that was randomly selected:

<i>"The Piaggio P.IX, or Piaggio Stella P.IX, was an Italian nine-cylinder radial aircraft engine produced by Rinaldo Piaggio S.p.A. Based on the Gnome-Rhône 9K, the engine was rated at 600 hp (447 kW). Production was used to power a number of other aircraft developed in Italy. The main users were the Savoia-Marchetti SM.81 transport and the IMAM Ro.37bis, the main reconnaissance aircraft in the Regia Aeronautica during the Second Italo-Ethiopian War, Spanish Civil War and Second World War, but the engine was also used by other designs, including the prototype Savoia-Marchetti SM.79. Piaggio acquired a license from Gnome et Rhône in 1925 for their engines derived from the Bristol Jupiter and, in 1933, brought out a developed version, created under the direction of engineer Renzo Spolti. The engine had nine cylinders and was therefore named P.IX. It was one of a range of Piaggio radial engines named Stella, or Star, all based on the same radial design. The engine had cylinders that had steel barrels and aluminium heads. Aluminium alloy pistons were connected to a split crankshaft via articulated connecting rods. The valves were enclosed. Each cylinder retained the same bore and stroke as the Gnome-Rhône 9K, 146 millimetres (5.7 in) and 165 mm (6.5 in) respectively. However, it was more powerful and was rated at 600 horsepower (447 kW) when fitted with a supercharger. The engine was used to power aircraft that served during the Second Italo-Ethiopian War, Spanish Civil War and Second World War, including one hundred and forty Savoia-Marchetti SM.81s, a Regia Aeronautica transport, and the majority of the production of the IMAM Ro.37bis reconnaissance aircraft. Most had retired by 1943. P.IX R. Normally aspirated and geared. P.IX R.C. Supercharged and geared. P.IX R.C.10 Supercharged and geared, rated at 1,000 m (3,300 ft). P.IX R.C.40 Supercharged and geared, rated at 4,000 m (13,000 ft). CANT Z.504 CANT Z.506 prototype Caproni Ca.131 Caproni Ca.132 Caproni Ca.301 Caproni Ca.305 Caproni CH.1 IMAM Ro.37bis IMAM Ro.43 prototype Piaggio P.10bis Piaggio P.16 Savoia-Marchetti S.73 Savoia-Marchetti SM.79 prototype Savoia-Marchetti SM.81 Data from Piaggio, 1939 Type: 9-cylinder, single row, air cooled radial engine Bore: 146 mm (5.7 in) Stroke: 165 mm (6.5 in) Displacement: 24.9 L (1,519 in3) Length: 1,050 mm (41 in) Diameter: 1,408 mm (55.4 in) Dry weight: 430 kg (950 lb) Valvetrain: 2 x overhead valves per cylinder operated by rockers and pushrods Supercharger: Centrifugal compressor Fuel type: 87 Octane petrol Cooling system: Air-cooled Power output: Take-off: 610 hp (455 kW) at 2,100 rpm Cruise: 600 hp (450 kW) at 2,100 rpm at 4,000 m (13,000 ft) Compression ratio: 6.0:1 Related development Gnome-Rhône 9K Comparable engines Alfa Romeo 125 Nakajima Kotobuki Pratt & Whitney R-1340 Wasp Related lists List of aircraft engines Abate, Rosario; Alegi, Gregory; Apostollo, Giorgio (1992)."</i>

The final ~100 words are essentially a list of technical componenents. As we we are going through wikipedia articles including engineering and science this may be commonplace, however I want to keep these categories for a diverse corpus. I therefore want a quality check function.

In [3]:
# define quality check function to look for both short sentences and sentences with too many numberic terms both these need generous
# thresholds as typical wiki articles have lots of numbers, i.e. "From north to south, there is one span of 197 feet 2 inches (60.10 m), 
# eight of 260 feet (79 m), a swing span of 280 feet (85 m), seven spans of 200 feet (61 m), and one of 196 feet 6 inches (59.89 m). 
# The northernmost two and southernmost two spans are primarily over land; the end spans cross over Broad Street in Perryville and North
# Union Avenue in Havre de Grace. The bridge carries two tracks of the Northeast Corridor rail line. As of 2024, it is used by up to 90 Amtrak 
# intercity trains and 14 MARC Penn Line commuter trains per day. Maximum speed over the bridge is 90 miles per hour (140 km/h), slower than the 
# 135 miles per hour (217 km/h)"

# NB decided to change this. very hard to determine an ideal sentence length or amount of numbers. better indicator would be number of colons, 
# and section headings, i.e. "Fuel type:" additionally because this would initially have had \n newline tags to seperate content which have now 
# been removed. Also this /n tag removal results in long 'sentences' so i iwll incorporate a max sentence length to catch these - this is for 
# wiki only, as some gutenberg books particularly old ones have long sentences

def quality_check(text):

    # split text into sentences
    sentences = sent_tokenize(text)

    # reject if any sentence > 100 words
    if any(len(sentence.split()) > 100 for sentence in sentences):
        return False

    # reject if > 10 colons - this catches some tables, specifications etc.
    if text.count(":") > 10:
        return False

    # This will catch labels under 30 words, i.e. "Fuel type:" or "Engine:"
    # and allow up to 5 of these in a text
    if len(re.findall(r"\b[A-Z][A-Za-z0-9 -]{1,30}:", text)) > 5:
        return False

    # if no flags triggered return true
    return True

In [17]:
# now we can actually get the articles

documents = []

for topic, source_page in topic_sources.items():

    print("\nCollecting: ", topic)

    # get list source
    source_text = get_source_page_wiki_text(source_page)

    # get titles
    titles = get_wiki_titles(source_text)

    print("Possible articles: ", len(titles))

    # want to collect a random sample therefore randomise order
    random.shuffle(titles)

    # initialise counter
    total = 0

    for title in titles:

        # stop at 80 (80 per topic)
        if total == 80:
            break

        if ":" in title:
            continue

        # get article text from the title...
        text = get_article_text(title)
        # ...clean it...
        text = clean_text(text)
        
        # if the long word flag > 100 breaks in the clean text function this allows it to continue
        if text is None:
            continue

        # run quality check to reject specification pages / manuals etc.
        if not quality_check(text):
            continue

        # get sample if enough words
        sample = get_sample(text)

        if sample is None:
            continue

        documents.append({
            "source":     "Wikipedia",
            "topic":      topic,
            "title":      title,
            "word_count": len(sample.split()),
            "text":       sample
        })

        # incrment counter
        total += 1

# convert to df
wiki_df = pd.DataFrame(documents)


Collecting:  Engineering and technology
Possible articles:  3151

Collecting:  History
Possible articles:  3059

Collecting:  Natural sciences
Possible articles:  4508

Collecting:  Sports and recreation
Possible articles:  5731

Collecting:  Social sciences and society
Possible articles:  2705


In [19]:
# wiki_df

In [22]:
# test
# print(wiki_df.loc[205, "text"])

In [23]:
# add document ids
wiki_df["document_id"] = [

    # +1 as dont want '0', 4 digits as up to 8000 ultimately
    f"WIKI_{i+1:04d}"

    for i in range(len(wiki_df))
]

In [24]:
# specify df column order inline with other sets
wiki_df = wiki_df[
    [
        "document_id",
        "source",
        "topic",
        "text",
        "word_count"
    ]
]

In [25]:
wiki_df.shape

(400, 5)

In [26]:
wiki_df["topic"].value_counts()

topic
Engineering and technology     80
History                        80
Natural sciences               80
Sports and recreation          80
Social sciences and society    80
Name: count, dtype: int64

In [27]:
wiki_df["word_count"].value_counts().sort_index()

word_count
368     1
377     1
400    23
401    23
402    22
403    15
404    17
405    14
406    20
407    16
408    16
409     8
410    14
411    16
412    18
413    10
414    12
415    13
416    10
417    10
418    12
419     9
420     9
421     9
422     9
423     8
424     6
425     7
426     5
427     4
428     4
429     5
430     2
431     5
432     2
433     4
434     4
436     1
437     2
438     1
439     2
440     1
441     1
442     4
443     1
445     2
451     1
498     1
Name: count, dtype: int64

In [32]:
# INSPECT LONG ONES
pd.set_option("display.max_colwidth", None)
wiki_df[wiki_df["word_count"] == 498][["document_id", "text", "word_count"]]

,document_id,text,word_count
351,WIKI_0352,"Davis v. Ayala, 576 U.S. 257 (2015), was a case in which the United States Supreme Court upheld a death sentence of a Hispanic defendant despite the fact that all Blacks and Hispanics were rejected from the jury during the defendant's trial. The case involved a habeas corpus petition submitted by Hector Ayala, who was arrested and tried in the late 1980s for the alleged murder of three individuals during an attempted robbery of an automobile body shop in San Diego, California in April 1985. At trial, the prosecution used peremptory challenges to strike all Black and Hispanic jurors who were available for jury service. The trial court judge allowed the prosecution to explain the basis for the peremptory challenges outside the presence of Ayala's counsel and did not give the defense an opportunity to respond. Ayala was ultimately sentenced to death, but he filed several appeals challenging the constitutionality of the trial court's decision to exclude his counsel from the hearings. In a 5–4 opinion written by Justice Samuel Alito, the Supreme Court held that even if the trial court committed error, the error was harmless and that Ayala did not suffer any actual prejudice. Justice Sonia Sotomayor wrote a dissenting opinion in which she argued that Ayala's sentence should be reversed because the exclusion of Ayala's counsel from the hearings ""substantially influenced the outcome"" of the case. Additionally, Justice Anthony Kennedy wrote a separate concurring opinion in which he questioned the propriety of Ayala's placement in solitary confinement. In response, Justice Clarence Thomas wrote a one-paragraph concurring opinion in which he stated that Ayala's accommodations were ""far sight more spacious than those in which his victims ... now rest"". Commentators have described the case as ""important"" and note that will likely have a ""significant effect"" on similar cases in the future. However, some analysts have described the outcome as ""particularly unjust"". Justice Kennedy's concurring opinion also received significant coverage from the media, and some analysts suggested that solitary confinement may become a ""new battleground""for Justice Kennedy One commentator described Justice Kennedy's concurring opinion as ""the single most surprising and heartening development of the term"". In Batson v. Kentucky, the Supreme Court of the United States held that a violation of the Equal Protection Clause of the Fourteenth Amendment to the United States Constitution occurs when prosecutors use peremptory challenges to exclude jurors on the basis of race. If a defendant alleges that a prosecutor challenged a prospective juror on the basis of that juror's race (a process known as a ""Batson challenge""), trial courts will conduct a three-part analysis of the peremptory challenge in question: ""First, a defendant must make a prima facie showing that a peremptory challenge has been exercised on the basis of race; second, if that showing has been made, the prosecution must offer a race-neutral basis for striking the juror in question; and third, in light of the parties’ submissions, the trial court must determine whether the defendant has shown purposeful discrimination.""",498


In [33]:
# caused by long sentence of 100 words that took the text from 398 to 498. Otherwise article is fine so will trim manually and replace 
# this one to keep total word counts in line

In [42]:
new_text = """Davis v. Ayala, 576 U.S. 257 (2015), was a case in which the United States Supreme Court upheld a death sentence of a Hispanic defendant
despite the fact that all Blacks and Hispanics were rejected from the jury during the defendant's trial. The case involved a habeas corpus petition 
submitted by Hector Ayala, who was arrested and tried in the late 1980s for the alleged murder of three individuals during an attempted robbery of 
an automobile body shop in San Diego, California in April 1985. At trial, the prosecution used peremptory challenges to strike all Black and Hispanic 
jurors who were available for jury service. The trial court judge allowed the prosecution to explain the basis for the peremptory challenges outside 
the presence of Ayala's counsel and did not give the defense an opportunity to respond. Ayala was ultimately sentenced to death, but he filed several 
appeals challenging the constitutionality of the trial court's decision to exclude his counsel from the hearings. In a 5–4 opinion written by Justice 
Samuel Alito, the Supreme Court held that even if the trial court committed error, the error was harmless and that Ayala did not suffer any actual 
prejudice. Justice Sonia Sotomayor wrote a dissenting opinion in which she argued that Ayala's sentence should be reversed because the exclusion of 
Ayala's counsel from the hearings "substantially influenced the outcome" of the case. Additionally, Justice Anthony Kennedy wrote a separate 
concurring opinion in which he questioned the propriety of Ayala's placement in solitary confinement. In response, Justice Clarence Thomas wrote a 
one-paragraph concurring opinion in which he stated that Ayala's accommodations were "far sight more spacious than those in which his victims ... 
now rest". Commentators have described the case as "important" and note that will likely have a "significant effect" on similar cases in the future. 
However, some analysts have described the outcome as "particularly unjust". Justice Kennedy's concurring opinion also received significant coverage 
from the media, and some analysts suggested that solitary confinement may become a "new battleground"for Justice Kennedy One commentator described 
Justice Kennedy's concurring opinion as "the single most surprising and heartening development of the term". In Batson v. Kentucky, the Supreme Court 
of the United States held that a violation of the Equal Protection Clause of the Fourteenth Amendment to the United States Constitution occurs when 
prosecutors use peremptory challenges to exclude jurors on the basis of race."""

In [43]:
# problem now is that this new text will have \n tags so best to re-clean
new_text = clean_text(new_text)
len(new_text.split())

399

In [44]:
# replace old sample with manually cut dowen version
wiki_df.loc[wiki_df["document_id"] == "WIKI_0352", "text"] = new_text
# also need an up to date word count
wiki_df.loc[wiki_df["document_id"] == "WIKI_0352", "word_count"] = len(new_text.split())

In [48]:
# test
# wiki_df["word_count"].value_counts().sort_index()
# inspect same one
# pd.set_option("display.max_colwidth", None)
# wiki_df[wiki_df["document_id"] == "WIKI_0352"][["document_id", "text", "word_count"]]

In [49]:
# wiki_df.head(50)

In [50]:
# save this new DF as the final 400 wiki articles

wiki_df.to_csv(r"C:\Users\Jamie\Desktop\DSM500 Final Project\Data\Processed\wikipedia-dataset.csv", index = False)

___________

### Step 3: Gutenburg

Lots of problems caused by the old method, starting again completely. This time will use the publicly available csv found here:
https://www.gutenberg.org/ebooks/offline_catalogs.html Scroll to "The Project Gutenberg Catalog Metadata in Machine-Readable Format XML/RDF/CSV" then to "An Excel-compatible CSV spreadsheet of eBook metadata is also available here. This file is updated once a week". This file was downloaded 30/7/26.

In [97]:
gutenberg_data = pd.read_csv(r"C:\Users\Jamie\Desktop\DSM500 Final Project\Data\Raw\pg_catalog.csv")

In [98]:
gutenberg_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78999 entries, 0 to 78998
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Text#        78999 non-null  int64 
 1   Type         78999 non-null  object
 2   Issued       78999 non-null  object
 3   Title        78999 non-null  object
 4   Language     78999 non-null  object
 5   Authors      78834 non-null  object
 6   Subjects     78954 non-null  object
 7   LoCC         78743 non-null  object
 8   Bookshelves  75696 non-null  object
dtypes: int64(1), object(8)
memory usage: 5.4+ MB


In [99]:
# keep only english language books and check numbers to confirm some dropped
gutenberg_data = gutenberg_data[gutenberg_data["Language"] == "en"]
gutenberg_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 62587 entries, 0 to 78997
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Text#        62587 non-null  int64 
 1   Type         62587 non-null  object
 2   Issued       62587 non-null  object
 3   Title        62587 non-null  object
 4   Language     62587 non-null  object
 5   Authors      62439 non-null  object
 6   Subjects     62575 non-null  object
 7   LoCC         62375 non-null  object
 8   Bookshelves  60002 non-null  object
dtypes: int64(1), object(8)
memory usage: 4.8+ MB


In [100]:
# ~16000 dropped so it worked. Now to also only keep text docs (no picture / image files)
gutenberg_data = gutenberg_data[gutenberg_data["Type"] == "Text"]
gutenberg_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 61402 entries, 0 to 78997
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Text#        61402 non-null  int64 
 1   Type         61402 non-null  object
 2   Issued       61402 non-null  object
 3   Title        61402 non-null  object
 4   Language     61402 non-null  object
 5   Authors      61286 non-null  object
 6   Subjects     61393 non-null  object
 7   LoCC         61193 non-null  object
 8   Bookshelves  59743 non-null  object
dtypes: int64(1), object(8)
memory usage: 4.7+ MB


In [4]:
# Also worked, now need to specify the book categories we want. Have had problems already here - periodicals too messy, scienence too many 
# tables etc. fairy tales, despite being in English had multiple languages, so taking the stance that i am covering multiple types of writing
# and the primary purpose of this one is stories / books, not the subject of thos books, so will try to cover a range, but also restircivted to
# categories large enough that I can sample at least 80 books from them and have that sample random

In [102]:
# gutenberg_data["Subjects"].value_counts().head(50)

In [103]:
# Going from the top of this list, given there will be rejections I want at least 100 in a category, which leaves very few.
# I will avoid periodicals due to past experience with them both being too short and having multiple topics per periodical.
# Poetry will throw up punctuation and sentence detection errors and is not representative of 'normal' text.
# Going from the top this gives me a list of: Fiction, Science fiction, ENglish fiction 19th century, detective and mysetery stories
# and essays. As mentioned elsewhere, the project's main goal is tetx analysis and a vareity of topics is a bonus not a necessity.

In [104]:
subjects = {
    "Fiction": "Fiction",
    "Science fiction": "Science fiction",
    "English fiction": "English fiction -- 19th century",
    "Mystery": "Detective and mystery stories",
    "Essays": "Essays"
}

In [105]:
# As per metadata file from previous attempt, the books can be accessed in a variety of ways, i.e. book 1 can be accessed at:
# gutenberg.org/ebooks/1.html.images, or 1.epub3.images, or 1.epub.noimages, or 1.txt.utf-8 etc. so we need a function to go through these to find
# a chosen book

def get_book(etext):

    # list of possible file locations
    urls = [
        f"https://www.gutenberg.org/cache/epub/{etext}/pg{etext}.txt",
        f"https://www.gutenberg.org/files/{etext}/{etext}--0.txt",
        f"https://www.gutenberg.org/files/{etext}/{etext}.txt",
        f"https://www.gutenberg.org/ebooks/{etext}.html.images",
        f"https://www.gutenberg.org/ebooks/{etext}.epub.noimages",
        f"https://www.gutenberg.org/ebooks/{etext}.txt.utf-8",
    ]

    # for each url...
    for url in urls:

        try:
            
            # ...try to access it
            response = requests.get(url, timeout = 20)
            
            # if we can access it return the text of the response
            if response.status_code == 200:

                return response.text

        except:

            pass

    return None

In [106]:
# now that the previous function got the text of the book we now need to extract the ~ 400 word sample. But first to do this
# as not all books follow an identical pattern, i choose to cut the first 1500 words for contents / index / intro etc. - the 
# text before the book starts and randomly find a location from there to the end (minus 700 words in case the location chosen
# is the final one and to allow for long sentences etc). The sample is then taken from this random location.

gut_rng = random.Random(1985)

def get_sample(text, rng):

    # avoid crash when cleaning text returns a none
    if text is None:
        return None
        
    # convert text into words
    words = text.split()
    
    # if book less than 2200 words skip it entirely
    if len(words) < 2200:
        return None

    # now want to randomly choose a start position after first 1500 words.
    # unlike bbc news etc books can be long, ideally want samples from 
    # different parts of books, rather than all intros
    # avoid first 1500 words to skip contents, acknowledgements etc. and don't
    # permit last 500 as possible a position at very end if chosen and we still
    # want to be able to take the full sample
    start = rng.randint(1500, len(words) - 500)

    # chop off all text before the randommly chosen point
    remaining_text = " ".join(words[start:])
    # split taht remaining text into sentences
    sentences = sent_tokenize(remaining_text)
    # assuming the random location is not at the start of a sentence we now skip the 
    # first sentence as it is not whole and starts mid-way through
    sentences = sentences[1:]
    
    # initialise empty list of sentences and 0 wordcount
    chosen = []
    word_count = 0

    # loop through sentences...
    for sentence in sentences:
        # add each in turn to list
        chosen.append(sentence)
        # whilst also counting total words so far
        word_count += len(sentence.split())

        # once words hits > 400 stop
        if word_count > 400:

            break

    # put sentences back together
    sample = " ".join(chosen)
    # many old texts have long sentences and for this gutenberg batch this results in 
    # many > 500 word samples, so rejecting those that are > 450 here
    if len(sample.split()) > 450:
        return None
    # otherwise return the sample ~400 words
    return sample

In [107]:
# now need to put everything together, loop through the topics randomly choosing 80 books from each and 
# getting the samples from those books

# initialise array for books
all_books = []

for subject, keyword in subjects.items():

    print("\nProcessing: ", subject)

    # filter to books of correct topic
    subset = gutenberg_data[
        gutenberg_data["Subjects"].str.contains(
            keyword,
            case = False, 
            na = False
        )
    ]

    # shuffle entire subset
    possible = subset.sample(frac = 1, random_state = 1985)

    # initialise value for chosen books
    books_chosen = 0

    # go through rows of subset sample
    for _, row in possible.iterrows():

        # break out if we have selected 80 books
        if books_chosen == 80:

            break

        # get text of the book
        text = get_book(row["Text#"])
    
        # if it doesnt work continue to next book in possible subset
        if text is None:
            print("Download failed: ", row["Title"])
            continue
    
        # otherwise... clean up the text 
        text = clean_text(text)
        # skip book if cleaning rejected it
        if text is None:
            continue

        # extract ~400 words from book at random location
        sample = get_sample(text, gut_rng)
    
        # keep going through possible subset if sample doesn't work
        if sample is None:
            print("Too short: ", row["Title"])
            continue
    
        # if happy with sample add data to list of books
        all_books.append({
            "document_id": f"GUT_{len(all_books)+1:04d}",
            "source": "Gutenberg",
            "topic": subject,
            "title": row["Title"],
            "author": row["Authors"],
            "Text number": row["Text#"],
            "word_count": len(sample.split()),
            "text": sample
        })
    
        # increment total books chosen within possible subset
        books_chosen += 1
        
    print(subject, "finished.")    


Processing:  Fiction
Too short:  List, Ye Landsmen! A Romance of Incident
Too short:  The Disowned — Volume 08
Too short:  The nobles are coming
Fiction finished.

Processing:  Science fiction
Too short:  The Smiler
Too short:  The Moon and the Sun
Too short:  Two Plus Two Makes Crazy
Too short:  Touch the sky
Too short:  Two Timer
Too short:  Doorstep
Too short:  The Wedge
Too short:  Dear Nan Glanders
Too short:  Floor of Heaven
Too short:  Firegod
Too short:  Birthday Present
Science fiction finished.

Processing:  English fiction
Too short:  Pearl-Fishing; Choice Stories from Dickens' Household Words; Second Series
Too short:  Evan Harrington — Volume 2
Too short:  Jilted! Or, My Uncle's Scheme, Volume 1
Too short:  Alone on a Wide Wide Sea, Vol. 2 (of 3)
Too short:  Alice, or the Mysteries — Book 07
Too short:  A Strange Story — Volume 06
Too short:  Lord Ormont and His Aminta — Volume 1
Too short:  The Disowned — Volume 02
English fiction finished.

Processing:  Mystery
Too shor

In [108]:
gutenberg_df = pd.DataFrame(all_books)

In [125]:
# gutenberg_df.head()

In [110]:
print(gutenberg_df.shape)

(400, 8)


In [111]:
print(gutenberg_df["topic"].value_counts())

topic
Fiction            80
Science fiction    80
English fiction    80
Mystery            80
Essays             80
Name: count, dtype: int64


In [126]:
# print(gutenberg_df["word_count"].value_counts().sort_index())

In [115]:
# save this new DF as the final 400 gutenberg articles

gutenberg_df.to_csv(r"C:\Users\Jamie\Desktop\DSM500 Final Project\Data\Processed\gutenberg-dataset.csv", index = False)

_____________

### Step 4: Amazon reviews

I initially tried multiple different datasets here but the issue is that amazon reviews are typically quite short - even after sampling 100,000 plus
i had a maxlength of 280. I therefore decided to use the amazon fine food reviews dataset as the average review length is longer than typical for amazon. the negative here is that of course the topic is always food reviews, however as touched on for gutenberg, the primary purpose of this dataset is human written review data, not the topic of that data. 
https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews

In [149]:
amazon_reviews = pd.read_csv(r"C:\Users\Jamie\Desktop\DSM500 Final Project\Data\Raw\Amazon-food-reviews.csv")

In [150]:
# amazon_reviews.head()

In [151]:
amazon_reviews["word_count"] = (amazon_reviews["Text"].str.split().str.len())

In [152]:
amazon_reviews.describe()

,Id,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,word_count
count,568454.000000,568454.000000,568454.00000,568454.000000,5.684540e+05,568454.000000
mean,284227.500000,1.743817,2.22881,4.183199,1.296257e+09,80.264023
std,164098.679298,7.636513,8.28974,1.310436,4.804331e+07,79.455384
min,1.000000,0.000000,0.00000,1.000000,9.393408e+08,3.000000
25%,142114.250000,0.000000,0.00000,4.000000,1.271290e+09,33.000000
50%,284227.500000,0.000000,1.00000,5.000000,1.311120e+09,56.000000
75%,426340.750000,2.000000,2.00000,5.000000,1.332720e+09,98.000000
max,568454.000000,866.000000,923.00000,5.000000,1.351210e+09,3432.000000


In [153]:
# mean word count of 80 but key is that max is now actually 3432

In [154]:
# create subset dataframe where reviews are 350-450 words
amazon_long_reviews = amazon_reviews[
    (amazon_reviews["word_count"] >= 350)
    &
    (amazon_reviews["word_count"] <= 450)
]

print(amazon_long_reviews.shape)

(4139, 11)


In [155]:
# 4139 reviews to choose from so all good

In [157]:
# sample random 800 reviews for the 400 required to account for errors / unaccepted reviews
amazon_sample = amazon_long_reviews.sample(n=800, random_state = 1985)

In [158]:
# function to ectract sample text from review

def get_sample(text):

    # handle no text / bad sample
    if text is None:
        return None, None
        
    # need to split into sentences
    sentences = sent_tokenize(text)
    
    # initialise 0 words and open array to store sentences
    sentences_chosen = []
    word_count = 0

    # loop through sentences and add to array whilst counting words
    for sentence in sentences:
        sentences_chosen.append(sentence)
        word_count += len(sentence.split())
        
        # when words >= 400 break out
        if word_count >= 400:
            break
        
    # reconnect sentences to make sample
    sample = " ".join(sentences_chosen)

    return sample, len(sample.split())

In [159]:
# now need to use these functions to turn the chosen reviews into samples

# initialise array for ~400 word text samples
samples = []

for _, row in amazon_sample.iterrows():

    # clean the text first
    cleaned_review = clean_text(row["Text"])
    # if the clean didn't accept it carry on to next
    if cleaned_review is None:
        continue
            
    # use get sample function on the text in each row
    # if none then continue to next
    sample, word_count = get_sample(cleaned_review)
    if sample is None:
        continue
        
    # check this isn't a duplicate already seen
    if sample in [x["text"] for x in samples]:
        continue
    
    samples.append({
        "document_id": f"AMZ_{len(samples)+1:04d}",
        "source": "Amazon reviews",
        "topic": "Food reviews",
        "title": row["Summary"],
        "author": row["ProfileName"],
        "word_count": word_count,
        "text": sample
    })

    if len(samples) == 400:
        break

In [160]:
amazon_sampled_df = pd.DataFrame(samples)

In [161]:
amazon_sampled_df.shape

(400, 7)

In [163]:
amazon_sampled_df["word_count"].value_counts().sort_index()

word_count
311    1
343    1
344    2
346    5
347    3
      ..
425    1
426    1
430    1
436    2
438    1
Name: count, Length: 87, dtype: int64

In [167]:
amazon_sampled_df = amazon_sampled_df[
    [
        "document_id",
        "source",
        "topic",
        "text",
        "word_count"
    ]
]

In [170]:
# amazon_sampled_df.sample(10)

In [171]:
# all looks good, save to csv as final 400 amazon articles

amazon_sampled_df.to_csv(r"C:\Users\Jamie\Desktop\DSM500 Final Project\Data\Processed\amazon-dataset.csv", index = False)

_______

### Step 5: Reddit posts

Note to me. Looked at some datasets. As with Amazon reviews, many posts are < 400 words, and also re: the subject matter, whilst it would be nice to have variety, the most feasible way to do this is with the hugging face dataset as it has comments from 50 subreddits, problem is size.

In [261]:
reddit_data = load_dataset("HuggingFaceGECLM/REDDIT_comments", streaming = True)

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/45 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/46 [00:00<?, ?it/s]

In [262]:
# print(reddit_data)

In [263]:
subreddits = [
    "relationship_advice",
    "explainlikeimfive",
    "changemyview",
    "AskHistorians",
    "askscience"
]    

In [264]:
# need function to go through a subreddit and extract long comments, want a large size to a) compensate for the fact that
# not all comments will be long enough, and b) so that we can then choose 80 of these (per subtreddit) at random
# initially a 10,000 limit, but after pipeline this only left < 80 for askscience. want to randomly sample from a large
# pool, therefore inxreased to 100,000

def get_long_comments(dataset, subreddit, limit = 100000):

    # initialise empty list
    comments = []

    # loop through comments
    for i, comment in enumerate(dataset):

            text = comment["body"]
            # skip empty / deleted comments
            if not text or text in ["[deleted]", "[removed]"]:
                continue
            # skip bot comments
            if comment["author"] == "AutoModerator":
                continue

            # clean the text
            text = clean_text(text)
            # skip any comments rejected by cleaning
            if text is None:
                continue

            # NB: the following cleaning is intentionally not in the cleaning function
            # as it is reddit specific and I do not want to apply to other datasets
            # some samples are actually multiple responses seperated by ">".
            # I only want to get single source samples, so i will trim after >
            # so i am looking for x > y, x>y, x >y etc.
            text = re.split(r"\s+>\s+", text)[0]
            # then remove any more quote markers
            text = text.replace(">", " ")
            # also some comments include tables created with | - I remove these
            text = text.replace("|", " ")
            # remove additional whitespace caused by these
            text = re.sub(r"\s+", " ", text).strip()
            
            # count words
            word_count = len(text.split())

            # now we only want to keep those that have 350-450 words
            if 350 <= word_count <=450:
                # if right word count get the sample
                sample, sample_word_count = get_sample(text)
                if sample is None:
                    continue
                # skip if now < 350 words
                if sample_word_count < 350:
                    continue
                    
                # and add to comments list
                comments.append({
                    "text": sample,
                    "subreddit": subreddit,
                    "word_count": sample_word_count
                })

            if i >= limit:
                break

    return pd.DataFrame(comments)

In [265]:
# now need to run these functions to get 80 samples from each of the chosen subreddits

reddit_samples = []

for subreddit in subreddits:

    print("Getting from: ", subreddit)

    # get dataset
    data = reddit_data[subreddit]
    # get the long comments
    df = get_long_comments(data, subreddit)
    
    print("Available samples: ", len(df))

    # get 80 at random
    df = df.sample(n = 80, random_state = 1985)
    reddit_samples.append(df)

Getting from:  relationship_advice
Available samples:  729
Getting from:  explainlikeimfive
Available samples:  1155
Getting from:  changemyview
Available samples:  1590
Getting from:  AskHistorians
Available samples:  1058
Getting from:  askscience
Available samples:  572


In [266]:
# put all into one df
reddit_sampled_df = pd.concat(reddit_samples, ignore_index = True)

In [267]:
# add columns to bring inline with other datasets

reddit_sampled_df["document_id"] = [
    f"RED_{i+1:04d}"
    for i in range(len(reddit_sampled_df))
]

reddit_sampled_df["source"] = "Reddit"

# change category from subreddit to topic in line with others
reddit_sampled_df = reddit_sampled_df.rename(
    columns = {"subreddit": "topic"}
)

# and re-arrange
reddit_sampled_df = reddit_sampled_df[
    [
        "document_id",
        "source",
        "topic",
        "text",
        "word_count"
    ]
]

In [268]:
reddit_sampled_df.shape

(400, 5)

In [269]:
reddit_sampled_df["topic"].value_counts()

topic
relationship_advice    80
explainlikeimfive      80
changemyview           80
AskHistorians          80
askscience             80
Name: count, dtype: int64

In [270]:
reddit_sampled_df["word_count"].value_counts().sort_index()

word_count
350    4
351    6
352    7
353    5
354    9
      ..
431    1
432    1
436    3
449    1
450    1
Name: count, Length: 83, dtype: int64

Ran the above multiple times, randomly sampling 10 at a time, so easily ~50 and no errors, so happy now with this data too.

In [277]:
# all looks good, save to csv as final 400 reddit articles

reddit_sampled_df.to_csv(r"C:\Users\Jamie\Desktop\DSM500 Final Project\Data\Processed\reddit-dataset.csv", index = False)

____

### Step 6: Amalgamate datasets

Now want to put all data into one csv file for taking further from here as main dataset.

In [337]:
bbc_df =       pd.read_csv(r"C:\Users\Jamie\Desktop\DSM500 Final Project\Data\Processed\bbc_xsum_dataset.csv")
wiki_df =      pd.read_csv(r"C:\Users\Jamie\Desktop\DSM500 Final Project\Data\Processed\wikipedia-dataset.csv")
gutenberg_df = pd.read_csv(r"C:\Users\Jamie\Desktop\DSM500 Final Project\Data\Processed\gutenberg-dataset.csv")
amazon_df =    pd.read_csv(r"C:\Users\Jamie\Desktop\DSM500 Final Project\Data\Processed\amazon-dataset.csv")
reddit_df =    pd.read_csv(r"C:\Users\Jamie\Desktop\DSM500 Final Project\Data\Processed\reddit-dataset.csv")

In [338]:
print(bbc_df.columns)
print(wiki_df.columns)
print(gutenberg_df.columns)
print(amazon_df.columns)
print(reddit_df.columns)

Index(['document_id', 'source', 'topic', 'text', 'word_count'], dtype='object')
Index(['document_id', 'source', 'topic', 'text', 'word_count'], dtype='object')
Index(['document_id', 'source', 'topic', 'title', 'author', 'Text number',
       'word_count', 'text'],
      dtype='object')
Index(['document_id', 'source', 'topic', 'text', 'word_count'], dtype='object')
Index(['document_id', 'source', 'topic', 'text', 'word_count'], dtype='object')


In [339]:
# need all to be equal

gutenberg_df = gutenberg_df[
    [
        "document_id",
        "source",
        "topic",
        "text",
        "word_count"
    ]
]

In [340]:
final_df = pd.concat(
    [
        bbc_df,
        wiki_df,
        gutenberg_df,
        amazon_df,
        reddit_df
    ]
)

In [341]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2000 entries, 0 to 399
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   document_id  2000 non-null   object
 1   source       2000 non-null   object
 2   topic        2000 non-null   object
 3   text         2000 non-null   object
 4   word_count   2000 non-null   int64 
dtypes: int64(1), object(4)
memory usage: 93.8+ KB


In [342]:
final_df["source"].value_counts()

source
BBC               400
Wikipedia         400
Gutenberg         400
Amazon reviews    400
Reddit            400
Name: count, dtype: int64

In [343]:
# dont want 100 bbc followed by 100 wiki etc so randomise dataset

final_df = final_df.sample(frac = 1, random_state = 1985).reset_index(drop = True)

In [344]:
final_df.sample(10)

,document_id,source,topic,text,word_count
1623,RED_0311,Reddit,AskHistorians,"It is much harder to stage a revolt in North Korea than other countries or at least it was under Kim Jong-il. Summarizing the ideas of North Korean expert Bruce E. Bechtol, Jr. on May 23 2011 ""In North Korea there is no such thing as hierarchical organization of government. It is more like a guy in the center of a circle and all of the institutions feed into him. Nobody outside of Kim Jong Il controls more than a few staff members. In the army if you are a general commanding a core or a colonel commanding a regiment on your left shoulder you have a guy from the State Security Department overlooking that you are doing things politically correct and on your right shoulder you have a guy from the General Political Bureau making sure you are not going to try and overthrow the government. So unlike China and the Soviet Union where they have a political officer in every unit and in North Korea if you want to rebel you have to kill 2 guys and they answer to 2 separate chains of command. So Kim has built the power base in the country so nobody controls enough power so someone can come in and take control a coup d'etat."" There was a revolt in the 1990s in one of the military cores but Kim Jong Il went down and had them all purged. ""They have described three attempted coups that ended in bloodshed. In 1996, the Sixth Field Army was planning to revolt but the scheme was betrayed by a new commander. One or two plotters got away but Kim Jong-il's personal guards arrested senior officers and the Sixth Field Army's political commissars. On March 12, 1998, Kim suddenly announced a martial law ""exercise"" in Pyongyang and there was gunfire in the streets of the city. The Chinese later learned that two ministries were involved in a coup attempt, and that more than 20 ministerial-level officials were killed after it was crushed. In October 1999, a company of the Third Field Army rebelled in dissatisfaction over grain distribution during the nation's prolonged famine, which may have killed a million people."" You might also want to listen to this lecture on North Korea propoganda and culture ""B.R. Myers takes an in-depth look at North Korean society and the domestic propaganda to which its citizens are exposed. Myers argues that North Korea is a paranoid, military-dominated nationalist state """,411
267,GUT_0369,Gutenberg,Essays,"The world was all marble and blood and bronze, against a pitiless sky of pitiless gods. The world was Rome. No rule ever stood builded so impregnably from earth to stars--a merciless wall of power. Strength never planted upon the earth so stern a foot. Never was tyranny so invincibly bastioned to the cowed and conquered eye. And against all this marble and blood and bronze, what frail fantastic attack is this? What quaint expedition from fairy-land that comes so insignificantly against these battlements on which the Roman helmets catch the setting sun? A Star in the Sky. Some Shepherds from Judea. Three Wise Men from the East. Some Frankincense and Myrrh. A Mother and Child. Yes, a fairy-tale procession--but these are to conquer Rome, and that child at his mother's breast has but to speak three words, for all that marble and bronze to melt away: ""Love One Another."" It may well have seemed an almost ludicrous weapon--three gentle words. So one might attack a fortress with a flower. But Rome fell before them, for all that, and cruel as the world still is, so cruel a world can never be again. The history of Christianity from Christ to Tolstoi is the history of a ghost-story; and as Rome fell before the men it martyred, so Russia has been compelled at last to open its prison doors by the passive imperative of the three gentle words. Stone and iron are terribly strong to the eye and even to the arm of man, but they are as vapour before the breath of the soul. Many enthroned and magisterial authorities seem so much more important and powerful 

In [345]:
# all looks good, save to csv as final human-written dataset

final_df.to_csv(r"C:\Users\Jamie\Desktop\DSM500 Final Project\Data\Processed\final_human_dataset.csv", index = False)

___